# OLS diagnostics on simulated designs

Dr. Pavanam Thomas  
Copyright 2026 Dr. Pavanam Thomas

Two DGPs: a correctly specified linear mean, and a quadratic mean estimated with a linear projection. The second is misspecified by construction. Residual plots and an added-quadratic test are diagnostics, not identifying arguments.

Problem → formalization → assumptions → computation/estimation → validation → interpretation → limitations.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if (root / "src").exists():
    sys.path.insert(0, str(root / "src"))
elif (root.parent / "src").exists():
    sys.path.insert(0, str(root.parent / "src"))

from econci import dgp, ols, plots

## Correctly specified linear DGP

Estimand: the population slopes on $x_1$ and $x_2$. Assignment of $x$ is independent of the error in the DGP, so OLS is consistent for those slopes.

In [ ]:
linear = dgp.simulate_ols_linear(n=800, beta1=2.0, beta2=-0.5, seed=42)
fit = ols.fit_ols(linear, "y ~ x1 + x2", cov_type="HC1")
ols.coefficient_table(fit)

In [ ]:
print("VIF")
print(ols.vif_table(linear, ["x1", "x2"]))
print("Breusch-Pagan p-value", ols.breusch_pagan_test(fit)["lm_pvalue"])
infl = ols.influence_table(fit)
print("Share of observations with Cook's distance > 4/n:", float(infl["flag_cooks_4n"].mean()))

## Misspecified linear fit (omitted quadratic)

The mean is quadratic in $x$. Fitting $y \sim x$ omits curvature. Diagnostics should fail. HC standard errors do not repair a wrong conditional mean.

In [ ]:
miss = dgp.simulate_ols_omitted_quadratic(n=800, seed=42)
bad = ols.fit_ols(miss, "y ~ x")
print(ols.coefficient_table(bad))
print("Added quadratic test:", ols.added_quadratic_test(miss, "y", "x"))
print("RESET:", ols.ramsey_reset(bad, power=3))

In [ ]:
resid = ols.residual_diagnostics(bad)
plots.plot_residuals_vs_fitted(
    resid["fitted"].to_numpy(),
    resid["residual"].to_numpy(),
    "outputs/figures/notebook_ols_misspecified.png",
    title="Linear fit on a quadratic DGP",
)

## Limitations

A passing RESET test does not imply a causal interpretation of $\beta$. In the linear DGP the slopes are structural because the simulation made $x$ exogenous. That premise must be argued separately in observational work. Samples here are simulated; see `docs/data_policy.md`.